# Guía 09 — Entrenamiento, Comparación y Validación de Modelos de Machine Learning

> **Nota previa:** Sube `dataset_limpio.csv` dentro de `/content/crudos/` antes de ejecutar el notebook completo.

In [ ]:
# CREACIÓN DE CARPETAS
!mkdir -p crudos procesados documentos modelos
print("CARPETAS CREADAS")

In [ ]:
# IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN GLOBAL
from datetime import datetime
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor

# Configuración global
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de gráficos
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Librerías importadas correctamente")
print(f"   - numpy: {np.__version__}")
print(f"   - pandas: {pd.__version__}")
print(f"   - scikit-learn: {sklearn.__version__}")
print(f"\nSemilla de reproducibilidad: {RANDOM_STATE}")

In [ ]:
# DEFINICIÓN DE RUTAS
BASE_DIR = Path("/content/")
DATA_RAW = BASE_DIR / "crudos"
DATA_PROCESSED = BASE_DIR / "procesados"
DATA_DOCS = BASE_DIR / "documentos"
DATA_MODELS = BASE_DIR / "modelos"

for carpeta in [DATA_RAW, DATA_PROCESSED, DATA_DOCS, DATA_MODELS]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("RUTAS CONFIGURADAS!!")

In [ ]:
# CARGA DEL DATASET LIMPIO
limpio = DATA_RAW / "dataset_limpio.csv"
df = pd.read_csv(limpio)
print(df.columns.tolist())
print(f"Filas: {len(df)}")

In [ ]:
# PREPARACIÓN DE SERIE DIARIA, FEATURES Y SPLIT TEMPORAL 80/20
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])
df["festival_bin"] = df["festival"].map({"Yes": 1, "No": 0})

df_diario = (
    df.groupby("fecha_pedido")
    .agg(
        ventas_por_dia=("precio_pedido", "sum"),
        festival=("festival_bin", "max")
    )
    .reset_index()
    .sort_values("fecha_pedido")
    .reset_index(drop=True)
)

# Variables de rezago y media móvil sin fuga de datos
df_diario["lag_1"] = df_diario["ventas_por_dia"].shift(1)
df_diario["lag_7"] = df_diario["ventas_por_dia"].shift(7)
df_diario["media_movil_7"] = (
    df_diario["ventas_por_dia"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

# Dataset de modelado sin NaN en lags
df_modelo = df_diario.dropna(subset=["lag_1", "lag_7", "media_movil_7"]).copy()
df_modelo["dia_semana"] = df_modelo["fecha_pedido"].dt.dayofweek
df_modelo["mes"] = df_modelo["fecha_pedido"].dt.month

features = [
    "dia_semana",
    "mes",
    "festival",
    "lag_1",
    "lag_7",
    "media_movil_7"
]

X = df_modelo[features]
y = df_modelo["ventas_por_dia"]

# Split temporal cronológico 80/20
corte = int(len(df_modelo) * 0.80)
X_train = X.iloc[:corte]
X_test = X.iloc[corte:]
y_train = y.iloc[:corte]
y_test = y.iloc[corte:]
fecha_test = df_modelo["fecha_pedido"].iloc[corte:]

print("PREPARACIÓN COMPLETADA")
print("=" * 60)
print(f"Dataset de modelado: {len(df_modelo)} filas")
print(f"Entrenamiento: {len(X_train)} filas ({df_modelo['fecha_pedido'].iloc[0].date()} a {df_modelo['fecha_pedido'].iloc[corte - 1].date()})")
print(f"Prueba:        {len(X_test)} filas ({df_modelo['fecha_pedido'].iloc[corte].date()} a {df_modelo['fecha_pedido'].iloc[-1].date()})")

In [ ]:
# Catálogo de modelos candidatos
modelos = {
    "Baseline media":    DummyRegressor(strategy="mean"),
    "Regresión lineal":  LinearRegression(),
    "Árbol de decisión": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest":     RandomForestRegressor(random_state=RANDOM_STATE)
}

print("MODELOS EN EL CATÁLOGO")
print("=" * 55)
for i, nombre in enumerate(modelos.keys(), 1):
    print(f"  {i}. {nombre}")

In [ ]:
# Función de métricas y evaluación holdout
def metricas_regresion(y_real, y_pred):
    """Calcula MAE, RMSE y R²."""
    return {
        "MAE":  mean_absolute_error(y_real, y_pred),
        "RMSE": mean_squared_error(y_real, y_pred) ** 0.5,
        "R2":   r2_score(y_real, y_pred)
    }


def evaluar_holdout(nombre, estimador):
    """
    Entrena un modelo clonado, evalúa en train y test,
    y devuelve (modelo, fila_resultado, predicciones_test).
    """
    modelo = clone(estimador)
    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_test  = modelo.predict(X_test)

    mt = metricas_regresion(y_train, pred_train)
    mp = metricas_regresion(y_test,  pred_test)

    fila = {
        "Modelo":     nombre,
        "MAE_train":  mt["MAE"],
        "MAE_test":   mp["MAE"],
        "RMSE_test":  mp["RMSE"],
        "R2_test":    mp["R2"],
        "Brecha_MAE": mp["MAE"] - mt["MAE"]
    }

    return modelo, fila, pred_test


print("Funciones definidas correctamente")

In [ ]:
# Entrenar y evaluar todos los modelos (holdout)
resultados_holdout = []
modelos_entrenados = {}
predicciones       = {}

for nombre, estimador in modelos.items():
    modelo, fila, pred = evaluar_holdout(nombre, estimador)
    modelos_entrenados[nombre] = modelo
    predicciones[nombre]       = pred
    resultados_holdout.append(fila)

tabla_holdout = pd.DataFrame(resultados_holdout)

print("RESULTADOS EN HOLDOUT (sin ordenar)")
print("=" * 75)
print(tabla_holdout.round(2).to_string(index=False))

In [ ]:
# Ordenar por MAE_test y mejora vs baseline
mae_baseline = tabla_holdout.loc[
    tabla_holdout["Modelo"] == "Baseline media", "MAE_test"
].iloc[0]

tabla_holdout["Mejora_vs_baseline_pct"] = (
    (mae_baseline - tabla_holdout["MAE_test"]) / mae_baseline * 100
)

tabla_holdout = tabla_holdout.sort_values("MAE_test").reset_index(drop=True)

print(f"MAE del baseline: ${mae_baseline:,.2f}")
print("\nTABLA COMPARATIVA (ordenada por MAE_test)")
print("=" * 90)
print(tabla_holdout.round(2).to_string(index=False))

In [ ]:
# Diagnóstico de sobreajuste (brecha train/test)
print("DIAGNÓSTICO DE SOBREAJUSTE")
print("=" * 75)

for _, row in tabla_holdout.iterrows():
    nombre   = row["Modelo"]
    brecha   = row["Brecha_MAE"]
    mae_tr   = row["MAE_train"]

    if mae_tr < 1:
        señal = "(ROJO) Posible sobreajuste severo (MAE_train ≈ 0)"
    elif brecha > mae_tr * 2:
        señal = "(NARANJA) Posible sobreajuste (brecha > 2×MAE_train)"
    elif brecha < 0:
        señal = "(AMARILLO) Brecha negativa (test más fácil que train)"
    else:
        señal = "(VERDE) Brecha razonable"

    print(f"\n{nombre}")
    print(f"   MAE_train: ${mae_tr:>8,.2f}")
    print(f"   MAE_test:  ${row['MAE_test']:>8,.2f}")
    print(f"   Brecha:    ${brecha:>8,.2f}")
    print(f"   → {señal}")

In [ ]:
# Crear TimeSeriesSplit y visualizar folds
tscv = TimeSeriesSplit(n_splits=5)

print("FOLDS DE VALIDACIÓN TEMPORAL")
print("=" * 75)
print(f"{'Fold':<6}{'Train inicio':<14}{'Train fin':<14}{'Val inicio':<14}{'Val fin':<14}")

for fold, (idx_train, idx_val) in enumerate(tscv.split(X_train), 1):
    print(
        f"{fold:<6}"
        f"{idx_train[0]:<14}"
        f"{idx_train[-1]:<14}"
        f"{idx_val[0]:<14}"
        f"{idx_val[-1]:<14}"
    )

print(f"\nTotal de filas en X_train: {len(X_train)}")
print(f"Filas por fold de validación: ~{len(X_train) // (tscv.get_n_splits() + 1)}")

In [ ]:
# Función de validación temporal
def validar_temporal(nombre, estimador):
    """Aplica TimeSeriesSplit a un estimador y devuelve métricas por fold."""
    filas = []

    for fold, (idx_tr, idx_val) in enumerate(tscv.split(X_train), 1):
        X_tr  = X_train.iloc[idx_tr]
        X_val = X_train.iloc[idx_val]
        y_tr  = y_train.iloc[idx_tr]
        y_val = y_train.iloc[idx_val]

        m = clone(estimador)
        m.fit(X_tr, y_tr)

        pred_tr  = m.predict(X_tr)
        pred_val = m.predict(X_val)

        filas.append({
            "Modelo":    nombre,
            "Fold":      fold,
            "MAE_train": mean_absolute_error(y_tr, pred_tr),
            "MAE_val":   mean_absolute_error(y_val, pred_val),
            "RMSE_val":  mean_squared_error(y_val, pred_val) ** 0.5,
            "R2_val":    r2_score(y_val, pred_val)
        })

    return pd.DataFrame(filas)


print("Función 'validar_temporal' definida correctamente")

In [ ]:
# Ejecutar validación cruzada
validaciones = []
for nombre, estimador in modelos.items():
    validaciones.append(validar_temporal(nombre, estimador))

df_cv = pd.concat(validaciones, ignore_index=True)

print("DETALLE DE VALIDACIÓN CRUZADA")
print("=" * 75)
print(df_cv.round(2).to_string(index=False))

In [ ]:
# Resumen por modelo (media y std)
resumen_cv = (
    df_cv
    .groupby("Modelo")
    .agg(
        MAE_cv_media   = ("MAE_val",    "mean"),
        MAE_cv_std     = ("MAE_val",    "std"),
        RMSE_cv_media  = ("RMSE_val",   "mean"),
        R2_cv_media    = ("R2_val",     "mean")
    )
    .reset_index()
    .sort_values("MAE_cv_media")
    .reset_index(drop=True)
)

print("RESUMEN DE VALIDACIÓN CRUZADA")
print("=" * 85)
print(resumen_cv.round(2).to_string(index=False))

In [ ]:
# Integrar holdout + CV y marcar candidatos
comparacion = tabla_holdout.merge(resumen_cv, on="Modelo", how="left")
comparacion = comparacion.sort_values("MAE_test").reset_index(drop=True)

# Criterios de selección
comparacion["Supera_baseline"] = comparacion["Mejora_vs_baseline_pct"] > 0
comparacion["CV_estable"]      = (
    comparacion["MAE_cv_std"] < comparacion["MAE_cv_media"] * 0.25
)

columnas_mostrar = [
    "Modelo", "MAE_test", "Mejora_vs_baseline_pct",
    "MAE_cv_media", "MAE_cv_std", "Brecha_MAE",
    "Supera_baseline", "CV_estable"
]

print("TABLA COMPARATIVA INTEGRADA")
print("=" * 110)
print(comparacion[columnas_mostrar].round(2).to_string(index=False))

In [ ]:
# Gráfico de barras MAE_test por modelo
graf = comparacion.sort_values("MAE_test")

fig, ax = plt.subplots(figsize=(9, 4))
colores = ['#e74c3c' if m == "Baseline media" else '#2c7bb6'
           for m in graf["Modelo"]]

bars = ax.bar(graf["Modelo"], graf["MAE_test"], color=colores, edgecolor='black')

# Etiquetas de valor sobre cada barra
for bar, val in zip(bars, graf["MAE_test"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"${val:,.0f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel("MAE en prueba ($)")
ax.set_title("Comparación de modelos", fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Estabilidad entre folds (una gráfica por modelo)
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for i, nombre in enumerate(df_cv["Modelo"].unique()):
    temp = df_cv[df_cv["Modelo"] == nombre]
    axes[i].plot(temp["Fold"], temp["MAE_val"],
                 marker='o', linewidth=2, color='#2c7bb6')
    axes[i].set_title(nombre, fontweight='bold')
    axes[i].set_xlabel("Fold temporal")
    axes[i].set_ylabel("MAE validación ($)")
    axes[i].grid(alpha=0.3, linestyle='--')
    axes[i].set_xticks(temp["Fold"].tolist())

plt.suptitle("Estabilidad entre folds por modelo", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Seleccionar candidato provisional
candidatos = comparacion[comparacion["Supera_baseline"]].copy()
candidato_ordenado = candidatos.sort_values("MAE_test")

nombre_candidato = candidato_ordenado.iloc[0]["Modelo"]
modelo_candidato = modelos_entrenados[nombre_candidato]
pred_candidato   = predicciones[nombre_candidato]

print("CANDIDATO PROVISIONAL")
print("=" * 55)
print(f"  Modelo: {nombre_candidato}")
print(f"  MAE_test: ${candidato_ordenado.iloc[0]['MAE_test']:,.2f}")
print(f"  RMSE_test: ${candidato_ordenado.iloc[0]['RMSE_test']:,.2f}")
print(f"  R²_test: {candidato_ordenado.iloc[0]['R2_test']:.4f}")
print(f"  Mejora vs baseline: {candidato_ordenado.iloc[0]['Mejora_vs_baseline_pct']:.1f}%")

print(f"\nModelos que superan al baseline:")
for _, row in candidato_ordenado.iterrows():
    print(f"  - {row['Modelo']}: ${row['MAE_test']:,.2f}")

In [ ]:
# Gráfico Real vs Predicho del candidato
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(fecha_test, y_test.values, label="Real",
        color='#2c7bb6', marker='o', linewidth=2, markersize=5)
ax.plot(fecha_test, pred_candidato, label="Predicción",
        color='#e74c3c', marker='s', linewidth=2,
        markersize=5, linestyle='--')

ax.set_title(f"Real vs Predicho — {nombre_candidato}",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas diarias ($)")
ax.legend(loc='upper left')
ax.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 errores más grandes del candidato
errores = pd.DataFrame({
    "fecha":    fecha_test.values,
    "real":     y_test.values,
    "predicho": pred_candidato
})

errores["error"]     = errores["real"] - errores["predicho"]
errores["error_abs"] = errores["error"].abs()

print("ESTADÍSTICAS DE ERROR")
print("=" * 55)
print(f"  Error promedio:  ${errores['error'].mean():,.2f}")
print(f"  Error mínimo:    ${errores['error'].min():,.2f}")
print(f"  Error máximo:    ${errores['error'].max():,.2f}")

print(f"\nTOP 5 ERRORES MÁS GRANDES")
print("=" * 75)
print(errores.nlargest(5, "error_abs").round(2).to_string(index=False))

In [ ]:
# Interpretación de variables del candidato
if hasattr(modelo_candidato, "coef_"):
    importancia = pd.DataFrame({
        "variable": features,
        "valor":    modelo_candidato.coef_
    }).sort_values("valor", key=lambda s: s.abs(), ascending=False)

    print(f"COEFICIENTES — {nombre_candidato}")
    print("=" * 65)
    print(importancia.round(4).to_string(index=False))
    print(f"\nIntercepto: ${modelo_candidato.intercept_:,.2f}")

elif hasattr(modelo_candidato, "feature_importances_"):
    importancia = pd.DataFrame({
        "variable": features,
        "valor":    modelo_candidato.feature_importances_
    }).sort_values("valor", ascending=False)

    print(f"FEATURE IMPORTANCE — {nombre_candidato}")
    print("=" * 65)
    print(importancia.round(4).to_string(index=False))
    print(f"\nSuma de importancias: {importancia['valor'].sum():.4f}")

else:
    importancia = pd.DataFrame()
    print("El modelo no expone coeficientes ni importancias.")

In [ ]:
# Gráfico de importancia de variables
if not importancia.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(importancia["variable"], importancia["valor"].abs(),
            color='#2c7bb6', edgecolor='black')
    ax.set_xlabel("Magnitud de la contribución")
    ax.set_title(f"Importancia de variables — {nombre_candidato}",
                 fontweight='bold')
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis='x', linestyle='--')
    plt.tight_layout()
    plt.show()
else:
    print("Sin datos de importancia para graficar.")

In [ ]:
# Matriz de decisión técnica
matriz = pd.DataFrame({
    "Criterio": [
        "Desempeño",
        "Generalización",
        "Estabilidad",
        "Interpretabilidad",
        "Costo",
        "Riesgo"
    ],
    "Pregunta": [
        "¿Supera claramente al baseline?",
        "¿La brecha train/test es razonable?",
        "¿Los folds son consistentes?",
        "¿Podemos explicar la salida?",
        "¿Tiempo y recursos adecuados?",
        "¿Errores graves en escenarios críticos?"
    ],
    "Valor observado": [
        f"{candidato_ordenado.iloc[0]['Mejora_vs_baseline_pct']:.1f}% mejora",
        f"Brecha ${candidato_ordenado.iloc[0]['Brecha_MAE']:,.2f}",
        f"Std ${candidato_ordenado.iloc[0]['MAE_cv_std']:,.2f}",
        "Coefs" if hasattr(modelo_candidato, "coef_") else "Feature importances",
        "Bajo (entrenamiento rápido en CPU)",
        "Bajo (sin sobreajuste severo)"
    ]
})

print("MATRIZ DE DECISIÓN TÉCNICA")
print("=" * 95)
print(matriz.to_string(index=False))

In [ ]:
# Exportar artefactos reproducibles
# 1) Tabla comparativa completa
ruta_comparacion = DATA_DOCS / "comparacion_modelos_proyecto_guia09.csv"
comparacion.to_csv(ruta_comparacion, index=False, encoding="utf-8")

# 2) Detalle de validación cruzada
ruta_cv = DATA_DOCS / "validacion_cruzada_proyecto_guia09.csv"
df_cv.to_csv(ruta_cv, index=False, encoding="utf-8")

# 3) Predicciones del candidato
ruta_pred = DATA_PROCESSED / "predicciones_proyecto_guia09.csv"
errores.to_csv(ruta_pred, index=False, encoding="utf-8")

# 4) Modelo candidato persistido
ruta_modelo = DATA_MODELS / "modelo_candidato_proyecto_guia09.joblib"
joblib.dump(modelo_candidato, ruta_modelo)

print("ARTEFACTOS EXPORTADOS")
print("=" * 75)
print(f"  - {ruta_comparacion.name}")
print(f"  - {ruta_cv.name}")
print(f"  - {ruta_pred.name}")
print(f"  - {ruta_modelo.name}")

In [ ]:
# Registrar versiones y metadata
metadata = {
    "fecha_ejecucion":  datetime.now().isoformat(),
    "python":           sys.version.split()[0],
    "scikit_learn":     sklearn.__version__,
    "pandas":           pd.__version__,
    "numpy":            np.__version__,
    "random_state":     RANDOM_STATE,
    "n_train":          len(X_train),
    "n_test":           len(X_test),
    "n_features":       len(features),
    "modelo_candidato": nombre_candidato
}

ruta_meta = DATA_DOCS / "metadata_proyecto_guia09.csv"
pd.DataFrame([metadata]).to_csv(ruta_meta, index=False, encoding="utf-8")

print("METADATA DEL EXPERIMENTO")
print("=" * 75)
for k, v in metadata.items():
    print(f"  {k}: {v}")
print(f"\nGuardado en: {ruta_meta.name}")

In [ ]:
# RESUMEN
print("=" * 75)
print("RESUMEN EJECUTIVO — GUÍA 9")
print("=" * 75)

print(f"\nMODELOS EVALUADOS: {len(modelos)}")
for nombre in modelos.keys():
    print(f"   • {nombre}")

print(f"\nMODELO CANDIDATO: {nombre_candidato}")
print(f"   MAE_test:           ${candidato_ordenado.iloc[0]['MAE_test']:,.2f}")
print(f"   RMSE_test:          ${candidato_ordenado.iloc[0]['RMSE_test']:,.2f}")
print(f"   R²_test:            {candidato_ordenado.iloc[0]['R2_test']:.4f}")
print(f"   Mejora vs baseline: {candidato_ordenado.iloc[0]['Mejora_vs_baseline_pct']:.1f}%")
print(f"   Estabilidad (std):  ${candidato_ordenado.iloc[0]['MAE_cv_std']:,.2f}")

print("\nARTEFACTOS GENERADOS:")
print(f"   • {ruta_comparacion.name}")
print(f"   • {ruta_cv.name}")
print(f"   • {ruta_pred.name}")
print(f"   • {ruta_modelo.name}")
print(f"   • {ruta_meta.name}")

print("\nSIGUIENTE PASO (Guía 10): optimizar hiperparámetros")
print("=" * 75)

In [ ]:
# VERIFICACIÓN FINAL
modelo_recargado = joblib.load(ruta_modelo) if ruta_modelo.exists() else None

verificaciones = {
    "Dataset preparado (793 filas)":     len(df_modelo) == 793,
    "X_train tiene 634 filas":           len(X_train) == 634,
    "X_test tiene 159 filas":            len(X_test) == 159,
    "4 modelos evaluados":               len(resultados_holdout) == 4,
    "MAE del baseline es positivo":      mae_baseline > 0,
    "Al menos 1 modelo supera baseline": bool(comparacion["Supera_baseline"].any()),
    "Candidato seleccionado":            nombre_candidato is not None,
    "Validación cruzada tiene 20 filas": len(df_cv) == 20,
    "Artefactos exportados":             ruta_modelo.exists(),
    "Modelo cargable con joblib":        modelo_recargado is not None
}

print("CHECKLIST DE VERIFICACIÓN")
print("=" * 60)
todo_ok = True
for descripcion, ok in verificaciones.items():
    icono = "SUCCESS" if ok else "FAILED"
    print(f"  {icono} {descripcion}")
    if not ok:
        todo_ok = False

print("\n" + "=" * 60)
if todo_ok:
    print("TODAS LAS VERIFICACIONES PASARON")
else:
    print("ALGUNAS VERIFICACIONES FALLARON — revisar celdas previas")